# Publicacao no HuggingFace Hub (bloco 3.2) — Kaggle

**Autor**: Massanori
**Descricao**: Publica os **3 modulos calibrados** (apenas `state_dict`) + os 3 `q_hat_*.json` no HuggingFace Hub, com model card e MANIFEST auditavel. Opcionalmente publica **1 volume de amostra** para o demo publico.

> **Privacidade**: sobe apenas `model_state_dict`. Nunca o `best.pt` inteiro (que carrega optimizer state). Atende ao aviso do guia (6.4) e a politica de anonimizacao.

## Pre-requisitos

- *Add Input* (4 obrigatorios + 2 opcionais para a amostra):
  1. `tcc-mri-resm-checkpoints` (best.pt do Grupo A)
  2. `tcc-mri-qr-checkpoints` (best.pt do Grupo B)
  3. `tcc-mri-qr-lesion-checkpoints` (best.pt do Grupo C)
  4. `tcc-mri-conformal-qhats` (3 JSONs do S5.7)
  5. (amostra) `tcc-mri-recons-varnet-brain-4x` (split test)
  6. (amostra) `tcc-mri-lesion-masks`
- *Add-ons -> Secrets*: crie um secret **`HF_TOKEN`** com um token de escrita do HuggingFace.

## Fluxo

1. Setup (clone do repo + `huggingface_hub`).
2. Token do HF a partir dos Kaggle Secrets.
3. Localiza os inputs.
4. **Dry-run** (extrai, valida round-trip, monta staging — nao sobe nada).
5. Upload real (depois de conferir o dry-run).


In [ ]:
# Celula 1 — Setup: clone repo + huggingface_hub
import os
import subprocess
import sys
from pathlib import Path

REPO_URL = "https://github.com/KR0N0S7/tcc-mri-uncertainty.git"
REPO_DIR = Path("/kaggle/working/tcc-mri-uncertainty")

if REPO_DIR.is_dir():
    subprocess.run(["git", "-C", str(REPO_DIR), "pull", "--ff-only"], check=False)
else:
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(REPO_DIR)], check=True)

os.chdir(REPO_DIR)
if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))

subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                "-r", str(REPO_DIR / "requirements.txt"), "huggingface_hub"], check=False)
subprocess.run(["git", "-C", str(REPO_DIR), "log", "-1", "--oneline"], check=False)
print("Setup OK")


In [ ]:
# Celula 2 — Token do HF (Kaggle Secrets) + repos de destino
# Preencha com o SEU usuario do HuggingFace:
HF_MODEL_REPO = "<usuario>/tcc-mri-uncertainty"     # repo de modelo (publico)
HF_SAMPLE_REPO = "<usuario>/tcc-mri-demo-sample"    # repo de amostra (dataset, PRIVADO por padrao)
PUBLISH_SAMPLE = False   # True para tambem subir 1 volume de amostra (leia o aviso de licenca)

# Token: NUNCA hardcode. Vem dos Kaggle Secrets (ou variavel de ambiente).
try:
    from kaggle_secrets import UserSecretsClient
    os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")
    print("HF_TOKEN carregado dos Kaggle Secrets.")
except Exception as e:
    if not os.environ.get("HF_TOKEN"):
        raise RuntimeError(
            "HF_TOKEN nao encontrado. Crie um secret 'HF_TOKEN' em Add-ons -> Secrets, "
            "ou defina a variavel de ambiente HF_TOKEN."
        ) from e
    print("HF_TOKEN obtido do ambiente.")


In [ ]:
# Celula 3 — Localiza os inputs Kaggle
def find(slug, predicate):
    roots = [Path("/kaggle/input") / slug]
    dd = Path("/kaggle/input/datasets")
    if dd.is_dir():
        roots += [u / slug for u in dd.iterdir() if u.is_dir()]
    for r in roots:
        if not r.is_dir():
            continue
        hit = predicate(r)
        if hit:
            return hit
    raise FileNotFoundError(f"Input '{slug}' nao encontrado/incompleto.")

def first_best_pt(root):
    hits = [p for p in root.rglob("best.pt") if ".ipynb_checkpoints" not in p.parts]
    return sorted(hits, key=lambda p: len(p.parts))[0] if hits else None

RESM = find("tcc-mri-resm-checkpoints", first_best_pt)
QR = find("tcc-mri-qr-checkpoints", first_best_pt)
QRL = find("tcc-mri-qr-lesion-checkpoints", first_best_pt)
QHATS = find("tcc-mri-conformal-qhats",
             lambda r: r if all((r / f"q_hat_{g}.json").is_file() for g in "ABC") else None)

print("ResM     :", RESM)
print("QR       :", QR)
print("QR-Lesion:", QRL)
print("q_hats   :", QHATS)

SAMPLE_ARGS = []
if PUBLISH_SAMPLE:
    recons = find("tcc-mri-recons-varnet-brain-4x",
                  lambda r: (r / "test") if (r / "test").is_dir() else None)
    masks = find("tcc-mri-lesion-masks",
                 lambda r: r if any(r.glob("*.pt")) else None)
    # escolhe 1 volume de teste que tenha mascara correspondente
    npz = None
    for cand in sorted((recons).glob("*.npz")):
        if (masks / f"{cand.stem}.pt").is_file():
            npz = cand
            break
    if npz is None:
        raise FileNotFoundError("Nenhum volume de teste com mascara correspondente.")
    SAMPLE_ARGS = ["--sample-npz", str(npz),
                   "--sample-mask", str(masks / f"{npz.stem}.pt"),
                   "--sample-repo-id", HF_SAMPLE_REPO]
    print("amostra  :", npz.name)


In [ ]:
# Celula 4 — DRY-RUN: extrai state_dicts, valida round-trip, monta staging (NAO sobe)
import sys, subprocess
cmd = [sys.executable, "scripts/publish_to_hf.py",
       "--resm", str(RESM), "--qr", str(QR), "--qr-lesion", str(QRL),
       "--qhats-dir", str(QHATS), "--repo-id", HF_MODEL_REPO,
       "--staging-dir", "/kaggle/working/hf_staging", "--dry-run"] + SAMPLE_ARGS
print(" ".join(cmd), "\n")
assert subprocess.run(cmd).returncode == 0, "Dry-run falhou — revise os logs acima."


In [ ]:
# Celula 5 — UPLOAD real (rode somente apos conferir o dry-run da Celula 4)
import sys, subprocess
cmd = [sys.executable, "scripts/publish_to_hf.py",
       "--resm", str(RESM), "--qr", str(QR), "--qr-lesion", str(QRL),
       "--qhats-dir", str(QHATS), "--repo-id", HF_MODEL_REPO,
       "--staging-dir", "/kaggle/working/hf_staging"] + SAMPLE_ARGS
# Para amostra publica (cuidado com a licenca fastMRI), adicione "--sample-public".
print(" ".join(cmd), "\n")
assert subprocess.run(cmd).returncode == 0, "Upload falhou — veja os logs acima."


## Proximo passo

1. Abra `notebooks/demo.ipynb` e, na **Celula 2**, preencha:
   - `HF_MODEL_REPO = "<usuario>/tcc-mri-uncertainty"`
   - `HF_SAMPLE_REPO = "<usuario>/tcc-mri-demo-sample"` (se publicou a amostra)
2. O demo passa a baixar os modelos/q_hats/amostra direto do Hub — clicavel por qualquer pessoa, sem acesso ao Kaggle.

> **Licenca da amostra**: o volume de amostra e derivado do fastMRI/fastMRI+. Confirme o Data Use Agreement antes de tornar o repo de amostra publico. Na duvida, mantenha-o privado/gated e compartilhe o acesso com a banca.
